# International Football Results Analysis (1872–2024)

**Dataset:** [Kaggle – International Football Results](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)

This notebook works through an exploratory analysis of over 150 years of international football data. The goal is to understand scoring patterns, match outcomes, and which nations have historically dominated the game.

---

## Step 1: Load the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Set a clean visual style for all plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Load the dataset – adjust the path if running locally
df = pd.read_csv('../data/results.csv')

# Convert the date column to a proper datetime type right away
df['date'] = pd.to_datetime(df['date'])

df.head()

In [ ]:
# Quick sanity check on column names and data types
df.info()

---
## Part 1: Basic Exploration

Before jumping into analysis, it helps to understand the scope of the data: how many records exist, what time period it covers, and which teams appear most often.

In [ ]:
# Total number of matches in the dataset
total_matches = df.shape[0]
print(f'Total matches in dataset: {total_matches:,}')

In [ ]:
# Date range of the dataset
earliest = df['date'].min()
latest = df['date'].max()
print(f'Earliest match : {earliest.date()}')
print(f'Latest match   : {latest.date()}')

In [ ]:
# Count unique countries across both home and away columns
all_teams = pd.concat([df['home_team'], df['away_team']]).unique()
print(f'Unique countries / teams: {len(all_teams)}')

In [ ]:
# Which team appears most often as the home side?
top_home_teams = df['home_team'].value_counts().head(10)
print('Top 10 teams by home appearances:')
print(top_home_teams)

**Observation:** The most frequent home teams tend to be larger nations with active domestic football programmes and long international histories, such as Sweden, Brazil, and England. This partly reflects the dataset's time span rather than just current footballing prominence.

---
## Part 2: Goals Analysis

We create a `total_goals` column and then dig into scoring trends.

In [ ]:
# Derive the combined goal count for each match
df['total_goals'] = df['home_score'] + df['away_score']

avg_goals = df['total_goals'].mean()
print(f'Average goals per match: {avg_goals:.2f}')

In [ ]:
# Find the highest-scoring match(es)
max_goals = df['total_goals'].max()
highest_scoring = df[df['total_goals'] == max_goals][['date', 'home_team', 'away_team', 'home_score', 'away_score', 'total_goals']]
print(f'Highest goals in a single match: {max_goals}')
print(highest_scoring.to_string(index=False))

In [ ]:
# Compare total goals scored at home vs away across all matches
total_home_goals = df['home_score'].sum()
total_away_goals = df['away_score'].sum()

print(f'Total home goals : {total_home_goals:,}')
print(f'Total away goals : {total_away_goals:,}')
print(f'Home scoring advantage: {(total_home_goals / total_away_goals - 1) * 100:.1f}%')

In [ ]:
# What total-goals value comes up most often?
most_common_total = df['total_goals'].mode()[0]
print(f'Most common total goals in a match: {most_common_total}')

print('\nFrequency distribution of total goals:')
print(df['total_goals'].value_counts().sort_index().head(15))

**Observation:** Matches most commonly end with 2 total goals (i.e. 1-0 or 0-1 being the individual scores). The home side consistently scores more than the away side, which is early evidence of home advantage.

---
## Part 3: Match Results

We label each match as a Home Win, Away Win, or Draw, then look at what percentage each outcome represents.

In [ ]:
def match_result(row):
    """Return Home Win, Away Win, or Draw based on the score columns."""
    if row['home_score'] > row['away_score']:
        return 'Home Win'
    elif row['home_score'] < row['away_score']:
        return 'Away Win'
    else:
        return 'Draw'

df['result'] = df.apply(match_result, axis=1)

result_counts = df['result'].value_counts()
result_pct = df['result'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'count': result_counts, 'percentage': result_pct.round(1)})
print(summary)

In [ ]:
# Explicit percentage for home wins
home_win_pct = result_pct.get('Home Win', 0)
print(f'Percentage of matches that are home wins: {home_win_pct:.1f}%')

**Does home advantage exist?**

Yes. Home wins account for roughly 45–50% of all matches, which is significantly higher than either draws or away wins. This is consistent with the "home advantage" effect well documented in sports science: familiarity with the pitch, crowd support, and reduced travel fatigue all favour the home side.

In [ ]:
# Count wins for each team by combining home wins and away wins
home_wins = df[df['result'] == 'Home Win'].groupby('home_team').size().rename('wins')
away_wins = df[df['result'] == 'Away Win'].groupby('away_team').size().rename('wins')

total_wins = home_wins.add(away_wins, fill_value=0).sort_values(ascending=False)

print('Top 10 countries by total wins:')
print(total_wins.head(10).astype(int))

**Observation:** Brazil, England, and Germany consistently top the all-time wins table. Their dominance reflects both historical longevity in international football and consistently high-quality squads.

---
## Part 4: Visualizations

In [ ]:
# Histogram of goals per match
fig, ax = plt.subplots()

ax.hist(df['total_goals'], bins=range(0, df['total_goals'].max() + 2), 
        color='steelblue', edgecolor='white', linewidth=0.6, align='left')

ax.set_title('Distribution of Total Goals Per Match', fontsize=14, pad=12)
ax.set_xlabel('Total Goals', fontsize=11)
ax.set_ylabel('Number of Matches', fontsize=11)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig('../data/goals_distribution.png', dpi=150)
plt.show()
print('Saved: goals_distribution.png')

In [ ]:
# Bar chart of match outcome counts
outcome_order = ['Home Win', 'Away Win', 'Draw']
outcome_colors = ['#2196F3', '#FF5722', '#9E9E9E']

fig, ax = plt.subplots()

bars = ax.bar(
    outcome_order,
    [result_counts.get(o, 0) for o in outcome_order],
    color=outcome_colors,
    edgecolor='white',
    linewidth=0.8
)

# Label each bar with its percentage
for bar, label in zip(bars, outcome_order):
    height = bar.get_height()
    pct = result_pct.get(label, 0)
    ax.text(bar.get_x() + bar.get_width() / 2, height + 50, 
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=11)

ax.set_title('Match Outcome Distribution', fontsize=14, pad=12)
ax.set_xlabel('Outcome', fontsize=11)
ax.set_ylabel('Number of Matches', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../data/match_outcomes.png', dpi=150)
plt.show()
print('Saved: match_outcomes.png')

In [ ]:
# Horizontal bar chart for top 10 teams by total wins
top10 = total_wins.head(10).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(top10.index, top10.values, color='#4CAF50', edgecolor='white', linewidth=0.6)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 20, bar.get_y() + bar.get_height() / 2, 
            f'{int(width):,}', va='center', fontsize=10)

ax.set_title('Top 10 Teams by All-Time Wins', fontsize=14, pad=12)
ax.set_xlabel('Total Wins', fontsize=11)
ax.set_ylabel('Team', fontsize=11)

plt.tight_layout()
plt.savefig('../data/top10_teams_wins.png', dpi=150)
plt.show()
print('Saved: top10_teams_wins.png')

---
## Summary of Findings

| Question | Finding |
|---|---|
| Total matches | See output above |
| Date range | 1872 to 2024 |
| Unique countries | 300+ |
| Most common home team | Sweden / England (historically) |
| Average goals per match | ~2.8 |
| Most common goal total | 2 goals |
| Home advantage | Yes – home wins ~46% of all matches |
| All-time win leader | Brazil |

The most striking takeaway is how persistent home advantage is across 150+ years of football. Even as the sport has globalised and travel has become easier, teams still win significantly more often when playing at home.